In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report
import time

train = pd.read_csv("/content/train_so.csv")
test = pd.read_csv("/content/test_so.csv")
val = pd.read_csv("/content/val_so.csv")

y_train = train["class"].values
y_test = test["class"].values
y_val = val["class"].values

X_train = train.drop(columns=["class"])
X_test = test.drop(columns=["class"])
X_val = val.drop(columns=["class"])

mms = MinMaxScaler()
X_train = mms.fit_transform(X_train)
X_test = mms.transform(X_test)
X_val = mms.transform(X_val)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
y_train = torch.tensor(y_train)
y_test = torch.tensor(y_test)
y_val = torch.tensor(y_val)

train_data = DataLoader(TensorDataset(X_train, y_train), batch_size=128, shuffle=True)
test_data = DataLoader(TensorDataset(X_test, y_test), batch_size=128)
val_data = DataLoader(TensorDataset(X_val, y_val), batch_size=128)

# DyT module code adapted from the official DyT implementation by Zhu et al.
# Source: https://jiachenzhu.github.io/DyT/
class DyT(nn.Module):
    def __init__(self, dim, init_alpha=0.5):
        super().__init__()
        self.alpha = nn.Parameter(torch.ones(1) * init_alpha)
        self.gamma = nn.Parameter(torch.ones(dim))
        self.beta = nn.Parameter(torch.zeros(dim))

    def forward(self, x):
        x = torch.tanh(self.alpha * x)
        return self.gamma * x + self.beta


class Encoder_Block(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.multi_attn = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, dropout=0.2,batch_first=True)
        self.dyt1 = DyT(embed_dim)
        self.pffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Linear(embed_dim * 4, embed_dim)
        )
        self.dyt2 = DyT(embed_dim)

    def forward(self, x):
        multi_attn, _ = self.multi_attn(x, x, x)
        x = self.dyt1(x + multi_attn)
        x_pffn = self.pffn(x)
        x = self.dyt2(x + x_pffn)
        return x

class K_Transformer(nn.Module):
    def __init__(self, input_dim, embed_dim=128, num_heads=16, num_blocks=8):
        super().__init__()
        self.to_embed = nn.Linear(input_dim, embed_dim)
        self.k_transformer = nn.Sequential(*[
            Encoder_Block(embed_dim, num_heads) for i in range(num_blocks)])
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64,2)
        )

    def forward(self, x):
        x = self.to_embed(x).unsqueeze(1)
        x = self.k_transformer(x)
        x = x.squeeze(1)
        x = self.mlp(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = K_Transformer(input_dim=X_train.shape[1]).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

train_start =time.time()
for epoch in range(20):
    model.train()
    overall_loss = 0
    for x,y in train_data:
        x,y=x.to(device),y.to(device)
        optimizer.zero_grad()
        output = model(x)
        loss = criterion(output, y)
        loss.backward()
        optimizer.step()
        overall_loss += loss.item()
    print(f"Epoch {epoch+1}/{20}. Loss: {overall_loss}")
train_end = time.time()
print(train_end-train_start)

model.eval()
predictions = []
answers = val["class"].values
test_start =time.time()
with torch.no_grad():
    for x,y in val_data:
        x=x.to(device)
        output = model(x)
        pred = torch.argmax(output, dim=1).cpu().numpy()
        predictions.extend(pred)
test_end =time.time()
print(test_end-test_start)
print(classification_report(answers, predictions, target_names=["Normal", "Malware"], digits=4))
torch.save(model.state_dict(), "best_k_dyt.pt")

Epoch 1/20. Loss: 127.69703494012356
Epoch 2/20. Loss: 81.23395773768425
Epoch 3/20. Loss: 69.91584435850382
Epoch 4/20. Loss: 68.3912567421794
Epoch 5/20. Loss: 68.66992004960775
Epoch 6/20. Loss: 67.73483280837536
Epoch 7/20. Loss: 67.14883040636778
Epoch 8/20. Loss: 67.07087865844369
Epoch 9/20. Loss: 66.76374448463321
Epoch 10/20. Loss: 66.4993677586317
Epoch 11/20. Loss: 48.391141440719366
Epoch 12/20. Loss: 32.703111693263054
Epoch 13/20. Loss: 31.16768140718341
Epoch 14/20. Loss: 30.15432421863079
Epoch 15/20. Loss: 29.91192327439785
Epoch 16/20. Loss: 29.063286264427006
Epoch 17/20. Loss: 28.997034227475524
Epoch 18/20. Loss: 28.320486476644874
Epoch 19/20. Loss: 28.56586054712534
Epoch 20/20. Loss: 28.08904157206416
131.36432576179504
0.3162527084350586
              precision    recall  f1-score   support

      Normal     0.9568    0.9943    0.9752      3160
     Malware     0.9903    0.9283    0.9583      1981

    accuracy                         0.9689      5141
   macro 

In [ ]:
model.load_state_dict(torch.load("best_k_dyt.pt"))
model.eval()
predictions = []
answers = test["class"].values
test_start =time.time()
with torch.no_grad():
    for x,y in test_data:
        x=x.to(device)
        output = model(x)
        pred = torch.argmax(output, dim=1).cpu().numpy()
        predictions.extend(pred)
test_end =time.time()
print(test_end-test_start)
print(classification_report(answers, predictions, target_names=["Normal", "Malware"], digits=4))

0.2116992473602295
              precision    recall  f1-score   support

      Normal     0.9564    0.9921    0.9739      3160
     Malware     0.9866    0.9278    0.9563      1981

    accuracy                         0.9673      5141
   macro avg     0.9715    0.9600    0.9651      5141
weighted avg     0.9680    0.9673    0.9671      5141



In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report
import time

train = pd.read_csv("/content/train_so.csv")
test = pd.read_csv("/content/test_so.csv")
val = pd.read_csv("/content/val_so.csv")

y_train = train["class"].values
y_test = test["class"].values
y_val = val["class"].values

X_train = train.drop(columns=["class"])
X_test = test.drop(columns=["class"])
X_val = val.drop(columns=["class"])

mms = MinMaxScaler()
X_train = mms.fit_transform(X_train)
X_test = mms.transform(X_test)
X_val = mms.transform(X_val)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
y_train = torch.tensor(y_train)
y_test = torch.tensor(y_test)
y_val = torch.tensor(y_val)

train_data = DataLoader(TensorDataset(X_train, y_train), batch_size=128, shuffle=True)
test_data = DataLoader(TensorDataset(X_test, y_test), batch_size=128)
val_data = DataLoader(TensorDataset(X_val, y_val), batch_size=128)

# DyT module code adapted from the official DyT implementation by Zhu et al.
# Source: https://jiachenzhu.github.io/DyT/
class DyT(nn.Module):
    def __init__(self, dim, init_alpha=0.3):
        super().__init__()
        self.alpha = nn.Parameter(torch.ones(1) * init_alpha)
        self.gamma = nn.Parameter(torch.ones(dim))
        self.beta = nn.Parameter(torch.zeros(dim))

    def forward(self, x):
        x = torch.tanh(self.alpha * x)
        return self.gamma * x + self.beta


class Encoder_Block(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.multi_attn = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, dropout=0.2,batch_first=True)
        self.dyt1 = DyT(embed_dim)
        self.pffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Linear(embed_dim * 4, embed_dim)
        )
        self.dyt2 = DyT(embed_dim)

    def forward(self, x):
        multi_attn, _ = self.multi_attn(x, x, x)
        x = self.dyt1(x + multi_attn)
        x_pffn = self.pffn(x)
        x = self.dyt2(x + x_pffn)
        return x

class K_Transformer(nn.Module):
    def __init__(self, input_dim, embed_dim=128, num_heads=16, num_blocks=8):
        super().__init__()
        self.to_embed = nn.Linear(input_dim, embed_dim)
        self.k_transformer = nn.Sequential(*[
            Encoder_Block(embed_dim, num_heads) for i in range(num_blocks)])
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64,2)
        )

    def forward(self, x):
        x = self.to_embed(x).unsqueeze(1)
        x = self.k_transformer(x)
        x = x.squeeze(1)
        x = self.mlp(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = K_Transformer(input_dim=X_train.shape[1]).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

train_start =time.time()
for epoch in range(20):
    model.train()
    overall_loss = 0
    for x,y in train_data:
        x,y=x.to(device),y.to(device)
        optimizer.zero_grad()
        output = model(x)
        loss = criterion(output, y)
        loss.backward()
        optimizer.step()
        overall_loss += loss.item()
    print(f"Epoch {epoch+1}/20. Loss: {overall_loss}")
train_end = time.time()
print(train_end-train_start)

model.eval()
predictions = []
answers = test["class"].values
test_start =time.time()
with torch.no_grad():
    for x,y in test_data:
        x=x.to(device)
        output = model(x)
        pred = torch.argmax(output, dim=1).cpu().numpy()
        predictions.extend(pred)
test_end =time.time()
print(test_end-test_start)
print(classification_report(answers, predictions, target_names=["Normal", "Malware"], digits=4,zero_division=0))

Epoch 1/20. Loss: 215.66484773159027
Epoch 2/20. Loss: 145.5744793266058
Epoch 3/20. Loss: 119.71304120123386
Epoch 4/20. Loss: 119.47574274241924
Epoch 5/20. Loss: 117.44815465807915
Epoch 6/20. Loss: 116.76610580086708
Epoch 7/20. Loss: 122.65437580645084
Epoch 8/20. Loss: 118.36769153177738
Epoch 9/20. Loss: 123.43535542488098
Epoch 10/20. Loss: 121.31120650470257
Epoch 11/20. Loss: 120.95855383574963
Epoch 12/20. Loss: 120.88601140677929
Epoch 13/20. Loss: 122.46115116775036
Epoch 14/20. Loss: 122.41287268698215
Epoch 15/20. Loss: 122.28874093294144
Epoch 16/20. Loss: 122.13417857885361
Epoch 17/20. Loss: 122.41070172190666
Epoch 18/20. Loss: 122.22577691078186
Epoch 19/20. Loss: 122.45094607770443
Epoch 20/20. Loss: 122.2096152305603
126.72625708580017
0.21206879615783691
              precision    recall  f1-score   support

      Normal     0.9020    0.8880    0.8949      3160
     Malware     0.8256    0.8460    0.8357      1981

    accuracy                         0.8718     

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report
import time

train = pd.read_csv("/content/train_so.csv")
test = pd.read_csv("/content/test_so.csv")
val = pd.read_csv("/content/val_so.csv")

y_train = train["class"].values
y_test = test["class"].values
y_val = val["class"].values

X_train = train.drop(columns=["class"])
X_test = test.drop(columns=["class"])
X_val = val.drop(columns=["class"])

mms = MinMaxScaler()
X_train = mms.fit_transform(X_train)
X_test = mms.transform(X_test)
X_val = mms.transform(X_val)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
y_train = torch.tensor(y_train)
y_test = torch.tensor(y_test)
y_val = torch.tensor(y_val)

train_data = DataLoader(TensorDataset(X_train, y_train), batch_size=128, shuffle=True)
test_data = DataLoader(TensorDataset(X_test, y_test), batch_size=128)
val_data = DataLoader(TensorDataset(X_val, y_val), batch_size=128)

# DyT module code adapted from the official DyT implementation by Zhu et al.
# Source: https://jiachenzhu.github.io/DyT/
class DyT(nn.Module):
    def __init__(self, dim, init_alpha=0.7):
        super().__init__()
        self.alpha = nn.Parameter(torch.ones(1) * init_alpha)
        self.gamma = nn.Parameter(torch.ones(dim))
        self.beta = nn.Parameter(torch.zeros(dim))

    def forward(self, x):
        x = torch.tanh(self.alpha * x)
        return self.gamma * x + self.beta


class Encoder_Block(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.multi_attn = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, dropout=0.2,batch_first=True)
        self.dyt1 = DyT(embed_dim)
        self.pffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Linear(embed_dim * 4, embed_dim)
        )
        self.dyt2 = DyT(embed_dim)

    def forward(self, x):
        multi_attn, _ = self.multi_attn(x, x, x)
        x = self.dyt1(x + multi_attn)
        x_pffn = self.pffn(x)
        x = self.dyt2(x + x_pffn)
        return x

class K_Transformer(nn.Module):
    def __init__(self, input_dim, embed_dim=128, num_heads=16, num_blocks=8):
        super().__init__()
        self.to_embed = nn.Linear(input_dim, embed_dim)
        self.k_transformer = nn.Sequential(*[
            Encoder_Block(embed_dim, num_heads) for i in range(num_blocks)])
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64,2)
        )

    def forward(self, x):
        x = self.to_embed(x).unsqueeze(1)
        x = self.k_transformer(x)
        x = x.squeeze(1)
        x = self.mlp(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = K_Transformer(input_dim=X_train.shape[1]).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

train_start =time.time()
for epoch in range(20):
    model.train()
    overall_loss = 0
    for x,y in train_data:
        x,y=x.to(device),y.to(device)
        optimizer.zero_grad()
        output = model(x)
        loss = criterion(output, y)
        loss.backward()
        optimizer.step()
        overall_loss += loss.item()
    print(f"Epoch {epoch+1}/20. Loss: {overall_loss}")
train_end = time.time()
print(train_end-train_start)

model.eval()
predictions = []
answers = test["class"].values
test_start =time.time()
with torch.no_grad():
    for x,y in test_data:
        x=x.to(device)
        output = model(x)
        pred = torch.argmax(output, dim=1).cpu().numpy()
        predictions.extend(pred)
test_end =time.time()
print(test_end-test_start)
print(classification_report(answers, predictions, target_names=["Normal", "Malware"], digits=4,zero_division=0))

Epoch 1/20. Loss: 116.90970996767282
Epoch 2/20. Loss: 74.93450970202684
Epoch 3/20. Loss: 69.3703269585967
Epoch 4/20. Loss: 68.28905348479748
Epoch 5/20. Loss: 67.74413350224495
Epoch 6/20. Loss: 45.480810387060046
Epoch 7/20. Loss: 33.47680089995265
Epoch 8/20. Loss: 32.3785445895046
Epoch 9/20. Loss: 30.96252854913473
Epoch 10/20. Loss: 30.4758444391191
Epoch 11/20. Loss: 29.523371858522296
Epoch 12/20. Loss: 29.230445064604282
Epoch 13/20. Loss: 29.1900584846735
Epoch 14/20. Loss: 28.89204759709537
Epoch 15/20. Loss: 28.17752860672772
Epoch 16/20. Loss: 28.444235417060554
Epoch 17/20. Loss: 28.379900574684143
Epoch 18/20. Loss: 27.733677349984646
Epoch 19/20. Loss: 27.40204499103129
Epoch 20/20. Loss: 27.378858096897602
127.18349051475525
0.20369839668273926
              precision    recall  f1-score   support

      Normal     0.9653    0.9861    0.9756      3160
     Malware     0.9770    0.9435    0.9599      1981

    accuracy                         0.9697      5141
   macro

In [ ]:
class Encoder_Block(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.multi_attn = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, dropout=0.2,batch_first=True)
        self.ln1 = nn.LayerNorm(embed_dim)
        self.pffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Linear(embed_dim * 4, embed_dim)
        )
        self.ln2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        multi_attn, _ = self.multi_attn(x, x, x)
        x = self.ln1(x + multi_attn)
        x_pffn = self.pffn(x)
        x = self.ln2(x + x_pffn)
        return x

class K_Transformer_LN(nn.Module):
    def __init__(self, input_dim, embed_dim=128, num_heads=16, num_blocks=8):
        super().__init__()
        self.to_embed = nn.Linear(input_dim, embed_dim)
        self.k_transformer = nn.Sequential(*[
            Encoder_Block(embed_dim, num_heads) for i in range(num_blocks)])
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64,2)
        )

    def forward(self, x):
        x = self.to_embed(x).unsqueeze(1)
        x = self.k_transformer(x)
        x = x.squeeze(1)
        x = self.mlp(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = K_Transformer_LN(input_dim=X_train.shape[1]).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

train_start =time.time()
for epoch in range(20):
    model.train()
    overall_loss = 0
    for x,y in train_data:
        x,y=x.to(device),y.to(device)
        optimizer.zero_grad()
        output = model(x)
        loss = criterion(output, y)
        loss.backward()
        optimizer.step()
        overall_loss += loss.item()
    print(f"Epoch {epoch+1}/{20}. Loss: {overall_loss}")
train_end = time.time()
print(train_end-train_start)

model.eval()
predictions = []
answers = test["class"].values
test_start =time.time()
with torch.no_grad():
    for x,y in test_data:
        x=x.to(device)
        output = model(x)
        pred = torch.argmax(output, dim=1).cpu().numpy()
        predictions.extend(pred)
test_end =time.time()
print(test_end-test_start)
print(classification_report(answers, predictions, target_names=["Normal", "Malware"], digits=4))

Epoch 1/20. Loss: 48.478274343535304
Epoch 2/20. Loss: 30.60143916681409
Epoch 3/20. Loss: 26.728181334212422
Epoch 4/20. Loss: 23.8803878063336
Epoch 5/20. Loss: 22.6541936150752
Epoch 6/20. Loss: 20.030914462171495
Epoch 7/20. Loss: 19.76897333189845
Epoch 8/20. Loss: 17.922166692558676
Epoch 9/20. Loss: 17.591241172514856
Epoch 10/20. Loss: 16.275149838998914
Epoch 11/20. Loss: 15.507869766559452
Epoch 12/20. Loss: 15.645321793854237
Epoch 13/20. Loss: 15.203664916101843
Epoch 14/20. Loss: 15.1236407100223
Epoch 15/20. Loss: 14.278279909398407
Epoch 16/20. Loss: 14.665831482037902
Epoch 17/20. Loss: 14.49618709157221
Epoch 18/20. Loss: 14.20052317227237
Epoch 19/20. Loss: 14.377894254168496
Epoch 20/20. Loss: 13.753474785480648
121.23663568496704
0.21207928657531738
              precision    recall  f1-score   support

      Normal     0.9846    0.9918    0.9882      3160
     Malware     0.9867    0.9753    0.9810      1981

    accuracy                         0.9854      5141
  

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
lr = LogisticRegression(max_iter=1000)
start=time.time()
lr.fit(X_train, y_train)
end = time.time()
print(end-start)
start=time.time()
predictions = lr.predict(X_test)
end = time.time()
print(end-start)
print(classification_report(answers, predictions, target_names=["Normal", "Malware"], digits=4))

gnb = GaussianNB()
start=time.time()
gnb.fit(X_train, y_train)
end = time.time()
print(end-start)
start=time.time()
predictions = gnb.predict(X_test)
end = time.time()
print(end-start)
print(classification_report(answers, predictions, target_names=["Normal", "Malware"], digits=4))

svc = SVC()
start=time.time()
svc.fit(X_train, y_train)
end = time.time()
print(end-start)
start=time.time()
predictions = svc.predict(X_test)
end = time.time()
print(end-start)
print(classification_report(answers, predictions, target_names=["Normal", "Malware"], digits=4))

0.32140088081359863
0.002557039260864258
              precision    recall  f1-score   support

      Normal     0.9502    0.9000    0.9244      3160
     Malware     0.8529    0.9248    0.8874      1981

    accuracy                         0.9096      5141
   macro avg     0.9016    0.9124    0.9059      5141
weighted avg     0.9127    0.9096    0.9102      5141

0.045526981353759766
0.0037457942962646484
              precision    recall  f1-score   support

      Normal     0.6208    0.9997    0.7659      3160
     Malware     0.9808    0.0257    0.0502      1981

    accuracy                         0.6244      5141
   macro avg     0.8008    0.5127    0.4080      5141
weighted avg     0.7595    0.6244    0.4901      5141

33.32762026786804
1.5572538375854492
              precision    recall  f1-score   support

      Normal     0.9528    0.9892    0.9707      3160
     Malware     0.9817    0.9218    0.9508      1981

    accuracy                         0.9632      5141
   macr

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64,2)
        )
    def forward(self, x):
        return self.mlp(x)

model = MLP(input_dim=X_train.shape[1]).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)


train_start =time.time()
for epoch in range(20):
    model.train()
    overall_loss = 0
    for x,y in train_data:
        x,y=x.to(device),y.to(device)
        optimizer.zero_grad()
        output = model(x)
        loss = criterion(output, y)
        loss.backward()
        optimizer.step()
        overall_loss += loss.item()
    print(f"Epoch {epoch+1}/{20}. Loss: {overall_loss}")
train_end = time.time()
print(train_end-train_start)

model.eval()
predictions = []
answers = test["class"].values
test_start =time.time()
with torch.no_grad():
    for x,y in test_data:
        x=x.to(device)
        out = model(x)
        pred = torch.argmax(out, dim=1).cpu().numpy()
        predictions.extend(pred)
test_end=time.time()
print(test_end-test_start)
print(classification_report(answers, predictions, target_names=["Normal", "Malware"], digits=4))

Epoch 1/20. Loss: 203.07423621416092
Epoch 2/20. Loss: 155.86214673519135
Epoch 3/20. Loss: 119.21235689520836
Epoch 4/20. Loss: 100.37783926725388
Epoch 5/20. Loss: 90.47469562292099
Epoch 6/20. Loss: 83.94984571635723
Epoch 7/20. Loss: 79.12067955732346
Epoch 8/20. Loss: 74.7860861569643
Epoch 9/20. Loss: 71.56668791174889
Epoch 10/20. Loss: 68.31327256560326
Epoch 11/20. Loss: 65.26805732399225
Epoch 12/20. Loss: 62.63932802528143
Epoch 13/20. Loss: 60.183503307402134
Epoch 14/20. Loss: 58.05158603191376
Epoch 15/20. Loss: 55.46765837818384
Epoch 16/20. Loss: 53.49505354464054
Epoch 17/20. Loss: 51.23122763633728
Epoch 18/20. Loss: 48.95712395012379
Epoch 19/20. Loss: 47.09750463813543
Epoch 20/20. Loss: 45.42410007864237
10.363712787628174
0.03529667854309082
              precision    recall  f1-score   support

      Normal     0.9535    0.9547    0.9541      3160
     Malware     0.9277    0.9258    0.9267      1981

    accuracy                         0.9436      5141
   macro